In [5]:
import os
from pathlib import Path
import pandas as pd
import numpy as np

# -------------------------------------------------------------------------
# Configurable File Paths
# -------------------------------------------------------------------------
STATIC_DATA_DIR = Path("../ignore_files/data/seperated/")
ACTIVITY_DATA_DIR = Path("../ignore_files/data/full/")
OUTPUT_DATA_DIR = Path("../ignore_files/data/full_calibrated/")

GRAVITY = 9.81

# -------------------------------------------------------------------------
# Device Key Alignment Mapping
# Maps Activity Dataset (Model, Device) -> Static Dataset Folder Name(s)
# -------------------------------------------------------------------------
DEVICE_ALIGNMENT_MAP = {
    ('gear', 'gear_1'): ['gear_1'],
    ('gear', 'gear_2'): ['gear_2'],
    ('lgwatch', 'lgwatch_1'): ['lg_1'],
    ('lgwatch', 'lgwatch_2'): ['lg_2'],
    ('nexus4', 'nexus4_1'): ['it-119', 'it-120', 'it-123', 'it-137', 'it-139', 'it-140', 'it-141', 'it-144'],
    ('nexus4', 'nexus4_2'): ['nexus4_2'],
    ('s3', 's3_1'): ['it-101'],
    ('s3', 's3_2'): ['it-103', 'it-108'],
    ('s3mini', 's3mini_1'): ['3Renault-AH'],
    ('s3mini', 's3mini_2'): ['3Renault-AH'],
    ('samsungold', 'samsungold_1'): ['it-116'],
    ('samsungold', 'samsungold_2'): ['it-118', 'it-126', 'it-133'],
}


def load_static_dataset_as_df(static_root):
    """
    Traverses static folder structure (Orientation/DeviceName/*.csv)
    and loads all static samples into a single DataFrame.
    """
    orientations = ['Phoneonback', 'Phoneonfront', 'Phoneonbottom', 'Phoneontop', 'Phoneonleft', 'Phoneonright']
    records = []

    print("--> Traversing static dataset files...")
    for orientation in orientations:
        orient_dir = static_root / orientation
        if not orient_dir.exists():
            continue

        for device_dir in orient_dir.iterdir():
            if not device_dir.is_dir():
                continue

            static_dev_name = device_dir.name.lower().strip()

            for csv_file in device_dir.glob("*.csv"):
                try:
                    df = pd.read_csv(csv_file)
                    if set(['x', 'y', 'z']).issubset(df.columns):
                        df['Device'] = static_dev_name
                        df['Orientation'] = orientation
                        records.append(df[['Device', 'Orientation', 'x', 'y', 'z']])
                except Exception as e:
                    print(f"Error reading {csv_file}: {e}")

    if not records:
        raise ValueError(f"No static dataset CSVs were successfully read from {static_root}")

    return pd.concat(records, axis=0)


def compute_device_calibration_robust(static_df, g=GRAVITY):
    """
    Calculates calibration parameters using 1-g vector magnitude centering,
    eliminating the dependency on having all 6 opposing orientation files.
    """
    static_calib = {}

    for device_id, group in static_df.groupby('Device'):
        # 1. Unit Detection: Check if values are in g's (~1.0) or m/s^2 (~9.81)
        raw_vals = group[['x', 'y', 'z']].values
        raw_mags = np.linalg.norm(raw_vals, axis=1)
        unit_scale = g if np.mean(raw_mags) < 3.0 else 1.0
        
        scaled_vals = raw_vals * unit_scale

        # 2. Estimate Zero-g Offset (b)
        # Compute mean per orientation, then average them to get axis bias
        orient_means = group.copy()
        orient_means[['x', 'y', 'z']] *= unit_scale
        mean_by_orient = orient_means.groupby('Orientation')[['x', 'y', 'z']].mean()

        # Bias is the centroid shift of the static orientation means
        b = mean_by_orient.mean(axis=0).values

        # 3. Estimate Scale Factor (s)
        # Scale factor is the ratio of measured static magnitude (after offset subtraction) to true g (9.81)
        centered_vals = scaled_vals - b
        measured_g = np.mean(np.linalg.norm(centered_vals, axis=1))
        
        s_factor = measured_g / g if measured_g > 0 else 1.0
        
        # Clamp scale factor to realistic MEMS bounds [0.85, 1.15]
        s_factor = np.clip(s_factor, 0.85, 1.15)
        s = np.array([s_factor, s_factor, s_factor])

        static_calib[device_id] = {
            'acc_bias': b,
            'acc_scale': s,
            'gyro_bias': np.zeros(3)
        }

    # Map activity (Model, Device) pairs to static parameters
    aligned_db = {}
    for (act_model, act_dev), static_folder_list in DEVICE_ALIGNMENT_MAP.items():
        biases, scales, gyro_biases = [], [], []

        for folder in static_folder_list:
            fname = folder.lower().strip()
            if fname in static_calib:
                biases.append(static_calib[fname]['acc_bias'])
                scales.append(static_calib[fname]['acc_scale'])
                gyro_biases.append(static_calib[fname]['gyro_bias'])

        if biases:
            aligned_db[(act_model.lower(), act_dev.lower())] = {
                'acc_bias': np.mean(biases, axis=0),
                'acc_scale': np.mean(scales, axis=0),
                'gyro_bias': np.mean(gyro_biases, axis=0),
            }

    print(f"✓ Built spherical 1-g calibration parameters for {len(aligned_db)} activity device pairs.")
    return aligned_db


def apply_calibration(file_path, output_dir, calib_db, is_gyroscope=False):
    """Calibrates an activity CSV file and exports it to output_dir."""
    if not file_path.exists():
        print(f"File not found: {file_path}")
        return

    print(f"\n--> Calibrating activity file: {file_path.name}")
    df = pd.read_csv(file_path)

    calibrated_groups = []
    matched_count = 0
    total_groups = 0

    for (model, device), group in df.groupby(['Model', 'Device']):
        total_groups += 1
        key = (str(model).lower().strip(), str(device).lower().strip())

        group = group.copy()
        raw_xyz = group[['x', 'y', 'z']].values

        if key in calib_db:
            matched_count += 1
            params = calib_db[key]

            if is_gyroscope:
                # Gyroscope calibration: subtract zero-drift bias vector
                calibrated_xyz = raw_xyz - params['gyro_bias']
            else:
                # Accelerometer calibration: (raw - bias) / scale
                calibrated_xyz = (raw_xyz - params['acc_bias']) / params['acc_scale']

            group[['x', 'y', 'z']] = calibrated_xyz
        else:
            print(f"  [Warning] Key not matched: Model='{model}', Device='{device}'")

        calibrated_groups.append(group)

    calibrated_df = pd.concat(calibrated_groups, axis=0).sort_index()

    output_dir.mkdir(parents=True, exist_ok=True)
    out_path = output_dir / f"Calibrated_{file_path.name}"
    calibrated_df.to_csv(out_path, index=False)

    print(f"✓ Matched & Calibrated {matched_count}/{total_groups} device groups.")
    print(f"✓ File written to: {out_path}")


# -------------------------------------------------------------------------
# Execution Entry Point
# -------------------------------------------------------------------------
if __name__ == "__main__":
    # 1. Load static files into a single DataFrame
    static_df = load_static_dataset_as_df(STATIC_DATA_DIR)

    # 2. Build unit-aware robust calibration database
    calib_db = compute_device_calibration_robust(static_df)

    # 3. Target activity files setup
    files_to_process = [
        ("Phones_accelerometer.csv", False),
        ("Phones_gyroscope.csv", True),
        ("Watch_accelerometer.csv", False),
        ("Watch_gyroscope.csv", True),
    ]

    # 4. Run calibration execution loop
    for fname, is_gyro in files_to_process:
        target_path = ACTIVITY_DATA_DIR / fname
        if not target_path.exists():
            alt_fname = fname.replace("accelerometer", "accelerator")
            target_path = ACTIVITY_DATA_DIR / alt_fname

        apply_calibration(target_path, OUTPUT_DATA_DIR, calib_db, is_gyroscope=is_gyro)

--> Traversing static dataset files...
✓ Built spherical 1-g calibration parameters for 12 activity device pairs.

--> Calibrating activity file: Phones_accelerometer.csv
✓ Matched & Calibrated 8/8 device groups.
✓ File written to: ../ignore_files/data/full_calibrated/Calibrated_Phones_accelerometer.csv

--> Calibrating activity file: Phones_gyroscope.csv
✓ Matched & Calibrated 6/6 device groups.
✓ File written to: ../ignore_files/data/full_calibrated/Calibrated_Phones_gyroscope.csv

--> Calibrating activity file: Watch_accelerometer.csv
✓ Matched & Calibrated 4/4 device groups.
✓ File written to: ../ignore_files/data/full_calibrated/Calibrated_Watch_accelerometer.csv

--> Calibrating activity file: Watch_gyroscope.csv
✓ Matched & Calibrated 4/4 device groups.
✓ File written to: ../ignore_files/data/full_calibrated/Calibrated_Watch_gyroscope.csv
